[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pyshka501/rl-textbook/blob/main/translations/ru/notebooks/ch17_open_problems_ru.ipynb)

<div style="background: linear-gradient(135deg, #1a0533 0%, #2d1b69 50%, #11998e 100%); padding: 40px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: #e0c3fc; font-size: 2.2em; margin: 0;">Глава 17. Открытые проблемы и новые направления</h1>
  <p style="color: #b0bec5; font-size: 1.1em; margin-top: 10px;">Decision Transformer · offline RL · model-based RL · сравнение online vs offline · фронтиры исследований</p>
</div>

Этот ноутбук обозревает ключевые открытые направления:
1. **Offline RL** — обучение по фиксированному датасету без взаимодействия со средой.
2. **Decision Transformer** — подход к RL через моделирование последовательностей.
3. **Сравнение** кривых обучения online vs offline.
4. **Model-based RL** — обучение модели динамики и планирование с её помощью.
5. **Обсуждение** открытых направлений исследований.

## Подготовка окружения

> **Colab/Kaggle**: запускается на бесплатном CPU. GPU не требуется.
> Все эксперименты используют минимальные синтетические среды — установка Gym/MuJoCo не нужна.
> Ожидаемое время: ~4 минуты от начала до конца.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict
from typing import List, Tuple, Dict, Optional
import random

np.random.seed(42)
random.seed(42)
print("Imports OK")

<div style="background: #1b1b3a; padding: 16px; border-radius: 8px; border-left: 4px solid #e0c3fc;">
  <h2 style="color: #e0c3fc; margin: 0;">Часть 1 — Offline RL: обучение по фиксированному датасету</h2>
</div>

В **offline (batch) RL** агент не может взаимодействовать со средой. Он должен учиться только по заранее собранному датасету.
Главная сложность — **сдвиг распределения**: стратегия может посещать пары состояние-действие, не охваченные данными.

Используем простой одномерный grid-world и сравниваем:
- Датасет, собранный **случайной стратегией** (низкое качество).
- Датасет, собранный **средней стратегией** (выше качество).
- Q-обучение в офлайн-режиме на каждом датасете.

In [ ]:
# Simple 1D GridWorld: states 0..N-1, goal at N-1
# Actions: 0=left, 1=right
# Reward: +10 at goal, -0.1 per step

class GridWorld1D:
    def __init__(self, n: int = 8):
        self.n = n
        self.goal = n - 1

    def step(self, state: int, action: int) -> Tuple[int, float, bool]:
        if action == 0:  # left
            next_state = max(0, state - 1)
        else:            # right
            next_state = min(self.n - 1, state + 1)
        done = (next_state == self.goal)
        reward = 10.0 if done else -0.1
        return next_state, reward, done

    def reset(self) -> int:
        return 0  # always start at left


def collect_dataset(env: GridWorld1D, policy_type: str,
                     n_transitions: int = 5000) -> List[Tuple]:
    """Collect a dataset using a given behaviour policy."""
    dataset = []
    state = env.reset()

    for _ in range(n_transitions):
        if policy_type == 'random':
            action = random.randint(0, 1)
        elif policy_type == 'medium':
            # Slightly right-biased
            action = 1 if random.random() < 0.75 else 0
        elif policy_type == 'expert':
            action = 1  # always right
        else:
            raise ValueError

        next_state, reward, done = env.step(state, action)
        dataset.append((state, action, reward, next_state, done))
        state = env.reset() if done else next_state

    return dataset


def offline_q_learning(env: GridWorld1D, dataset: List[Tuple],
                        n_epochs: int = 20, alpha: float = 0.1,
                        gamma: float = 0.95) -> np.ndarray:
    """Train Q-table offline on the fixed dataset (fitted Q-iteration style)."""
    Q = np.zeros((env.n, 2))
    n = len(dataset)

    for _ in range(n_epochs):
        random.shuffle(dataset)
        for (s, a, r, s2, done) in dataset:
            target = r if done else r + gamma * np.max(Q[s2])
            Q[s, a] += alpha * (target - Q[s, a])

    return Q


def evaluate_policy(env: GridWorld1D, Q: np.ndarray,
                     n_episodes: int = 100) -> float:
    """Greedy policy evaluation."""
    total = 0.0
    for _ in range(n_episodes):
        s = env.reset()
        for t in range(50):
            a = int(np.argmax(Q[s]))
            s, r, done = env.step(s, a)
            total += r
            if done:
                break
    return total / n_episodes


env = GridWorld1D(n=10)

results = {}
for policy_type in ['random', 'medium', 'expert']:
    dataset = collect_dataset(env, policy_type, 5000)
    Q = offline_q_learning(env, dataset)
    score = evaluate_policy(env, Q)
    results[policy_type] = score
    print(f"{policy_type:8s} dataset → mean return = {score:.2f}")

print("\nKey: higher quality datasets → better offline RL performance")

In [ ]:
# Online vs Offline learning curves

def online_q_learning(env: GridWorld1D, n_episodes: int = 500,
                       alpha: float = 0.1, gamma: float = 0.95,
                       epsilon: float = 0.2) -> List[float]:
    """Standard online Q-learning."""
    Q = np.zeros((env.n, 2))
    returns = []

    for ep in range(n_episodes):
        s = env.reset()
        ep_ret = 0.0
        for _ in range(50):
            if random.random() < epsilon:
                a = random.randint(0, 1)
            else:
                a = int(np.argmax(Q[s]))
            s2, r, done = env.step(s, a)
            Q[s, a] += alpha * (r + gamma * np.max(Q[s2]) * (1-done) - Q[s, a])
            s = s2; ep_ret += r
            if done: break
        returns.append(ep_ret)

    return returns


def offline_learning_curve(env: GridWorld1D, dataset: List[Tuple],
                             n_update_steps: int = 500) -> List[float]:
    """Offline Q-learning: evaluate after each epoch of updates."""
    Q = np.zeros((env.n, 2))
    batch_size = min(100, len(dataset))
    curve = []

    for step in range(n_update_steps):
        batch = random.sample(dataset, batch_size)
        for (s, a, r, s2, done) in batch:
            target = r if done else r + 0.95 * np.max(Q[s2])
            Q[s, a] += 0.05 * (target - Q[s, a])
        if step % 5 == 0:
            curve.append(evaluate_policy(env, Q, 50))

    return curve


online_curve = online_q_learning(env, 500)

dataset_random = collect_dataset(env, 'random', 5000)
dataset_medium = collect_dataset(env, 'medium', 5000)
dataset_expert = collect_dataset(env, 'expert', 5000)

offline_random = offline_learning_curve(env, dataset_random)
offline_medium = offline_learning_curve(env, dataset_medium)
offline_expert = offline_learning_curve(env, dataset_expert)


def smooth(x, w=10): return np.convolve(x, np.ones(w)/w, mode='valid')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('#0f1117')

# Online
ax = axes[0]
ax.set_facecolor('#0f1117')
ax.plot(smooth(online_curve, 20), color='#69f0ae', lw=2, label='Online Q-learning')
ax.set_xlabel('Episode'); ax.set_ylabel('Mean Return')
ax.set_title('Online Q-Learning', color='white')
ax.legend(facecolor='#1a1a2e', labelcolor='white')
for item in [ax.xaxis.label, ax.yaxis.label] + ax.get_xticklabels() + ax.get_yticklabels():
    item.set_color('white')

# Offline
ax = axes[1]
ax.set_facecolor('#0f1117')
x = np.arange(len(offline_random))
ax.plot(x, offline_random, color='#e94560', lw=2, label='Offline (random data)')
ax.plot(x, offline_medium, color='#ffb74d', lw=2, label='Offline (medium data)')
ax.plot(x, offline_expert, color='#69f0ae', lw=2, label='Offline (expert data)')
ax.set_xlabel('Update Step (×5)'); ax.set_ylabel('Mean Return')
ax.set_title('Offline Q-Learning by Dataset Quality', color='white')
ax.legend(facecolor='#1a1a2e', labelcolor='white', fontsize=9)
for item in [ax.xaxis.label, ax.yaxis.label] + ax.get_xticklabels() + ax.get_yticklabels():
    item.set_color('white')

plt.suptitle('Online vs Offline RL: 1D GridWorld', color='white', fontsize=14)
plt.tight_layout()
plt.show()

<div style="background: #1b1b3a; padding: 16px; border-radius: 8px; border-left: 4px solid #80deea;">
  <h2 style="color: #80deea; margin: 0;">Часть 2 — Decision Transformer (минимальная реализация)</h2>
</div>

**Decision Transformer** (Chen и др., 2021) рассматривает RL как задачу моделирования последовательностей.
Вход: `(оставшаяся отдача, состояние, действие, оставшаяся отдача, состояние, действие, ...)`, обусловленный **желаемой отдачей**.
Выход: следующее действие.

Реализуем минимальную версию на одномерном GridWorld через простой MLP (полный трансформер не нужен из-за CPU).

In [ ]:
# Decision Transformer key idea: condition on desired return-to-go (RTG)
# At inference: set RTG = desired_return, then greedily decode actions

# Collect expert trajectories with RTG labels
def collect_trajectories_with_rtg(env: GridWorld1D, n_eps: int = 500,
                                    policy_type: str = 'expert') -> List:
    """Return list of (states, actions, rtgs) for each trajectory."""
    trajs = []
    for _ in range(n_eps):
        states, actions, rewards = [], [], []
        s = env.reset()
        for _ in range(50):
            if policy_type == 'expert':
                a = 1  # go right
            elif policy_type == 'random':
                a = random.randint(0, 1)
            else:
                a = 1 if random.random() < 0.7 else 0

            s2, r, done = env.step(s, a)
            states.append(s); actions.append(a); rewards.append(r)
            s = s2
            if done: break

        # Compute return-to-go for each step
        rtgs = []
        cumulative = 0.0
        for r in reversed(rewards):
            cumulative += r
            rtgs.insert(0, cumulative)

        trajs.append((states, actions, rtgs))
    return trajs


def dt_train(trajs, n_iters: int = 2000):
    """Minimal DT: learn P(action | state, rtg) via logistic regression."""
    # Feature = [state/N, rtg/10.0]; target = action
    X, y = [], []
    for (states, actions, rtgs) in trajs:
        for s, a, rtg in zip(states, actions, rtgs):
            X.append([s / env.n, rtg / 10.0, (rtg / 10.0) ** 2])
            y.append(a)

    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.float32)

    # Simple 2-layer network via manual gradient descent
    n_hidden = 16
    W1 = np.random.randn(3, n_hidden) * 0.1
    b1 = np.zeros(n_hidden)
    W2 = np.random.randn(n_hidden, 1) * 0.1
    b2 = np.zeros(1)
    lr = 0.01

    losses = []
    idx = np.arange(len(X))
    batch = 64

    for it in range(n_iters):
        np.random.shuffle(idx)
        bi = idx[:batch]
        xb, yb = X[bi], y[bi, None]

        # Forward
        h = np.maximum(0, xb @ W1 + b1)  # ReLU
        logit = h @ W2 + b2
        prob = 1 / (1 + np.exp(-logit))
        loss = -np.mean(yb * np.log(prob + 1e-9) + (1-yb) * np.log(1-prob + 1e-9))

        # Backward
        dlogit = (prob - yb) / batch
        dW2 = h.T @ dlogit
        db2 = dlogit.sum(0)
        dh = dlogit @ W2.T * (h > 0)
        dW1 = xb.T @ dh
        db1 = dh.sum(0)

        W1 -= lr * dW1; b1 -= lr * db1
        W2 -= lr * dW2; b2 -= lr * db2
        if it % 100 == 0:
            losses.append(loss)

    def predict(state, rtg):
        x = np.array([[state / env.n, rtg / 10.0, (rtg / 10.0)**2]])
        h = np.maximum(0, x @ W1 + b1)
        logit = h @ W2 + b2
        prob_right = float(1 / (1 + np.exp(-logit[0, 0])))
        return 1 if prob_right > 0.5 else 0

    return predict, losses


trajs_expert = collect_trajectories_with_rtg(env, 500, 'expert')
trajs_random = collect_trajectories_with_rtg(env, 500, 'random')
trajs_mixed  = trajs_expert[:250] + trajs_random[:250]

dt_fn, losses = dt_train(trajs_expert + trajs_random, n_iters=3000)

print("Decision Transformer (minimal) training loss curve:")
fig, ax = plt.subplots(figsize=(7, 4))
ax.set_facecolor('#0f1117'); fig.patch.set_facecolor('#0f1117')
ax.plot(losses, color='#80deea', lw=2)
ax.set_xlabel('Iteration (×100)'); ax.set_ylabel('BCE Loss')
ax.set_title('DT Training Loss', color='white')
for item in [ax.xaxis.label, ax.yaxis.label] + ax.get_xticklabels() + ax.get_yticklabels():
    item.set_color('white')
plt.tight_layout(); plt.show()

In [ ]:
# DT inference: condition on different target RTGs
def dt_rollout(env, predict_fn, target_rtg: float, max_steps: int = 50) -> float:
    """Roll out the DT policy conditioned on a target return-to-go."""
    s = env.reset()
    total_r = 0.0
    rtg = target_rtg
    for _ in range(max_steps):
        a = predict_fn(s, rtg)
        s2, r, done = env.step(s, a)
        total_r += r
        rtg -= r
        s = s2
        if done: break
    return total_r


target_rtgs = [0, 2, 4, 6, 8, 9, 9.5]
dt_returns = []
for rtg in target_rtgs:
    avg_ret = np.mean([dt_rollout(env, dt_fn, rtg) for _ in range(50)])
    dt_returns.append(avg_ret)
    print(f"  Target RTG={rtg:.1f} → actual return = {avg_ret:.2f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.set_facecolor('#0f1117'); fig.patch.set_facecolor('#0f1117')
ax.plot(target_rtgs, dt_returns, 'o-', color='#80deea', lw=2, markersize=8)
ax.plot([0, 9.5], [0, 9.5], '--', color='gray', lw=1, label='Perfect (target=actual)')
ax.set_xlabel('Target Return-to-Go (RTG)')
ax.set_ylabel('Actual Mean Return')
ax.set_title('Decision Transformer: Conditioning on Target Return', color='white')
ax.legend(facecolor='#1a1a2e', labelcolor='white')
for item in [ax.xaxis.label, ax.yaxis.label] + ax.get_xticklabels() + ax.get_yticklabels():
    item.set_color('white')
plt.tight_layout(); plt.show()

<div style="background: #1b1b3a; padding: 16px; border-radius: 8px; border-left: 4px solid #ffcc02;">
  <h2 style="color: #ffcc02; margin: 0;">Часть 3 — Model-Based RL: учим динамику, потом планируем</h2>
</div>

**Model-based RL** (MBRL) поддерживает **обученную модель динамики** $\hat{p}(s' | s, a)$ и использует её для планирования.
Ключевые компоненты:
1. **Собираем переходы** из реальной среды.
2. **Обучаем модель динамики** на этих переходах.
3. **Генерируем синтетические роллауты** из модели.
4. **Обучаем Q-функцию** на синтетических + реальных данных.

In [ ]:
class DynamicsModel:
    """Simple tabular dynamics model P(s'|s,a) and R(s,a)."""

    def __init__(self, n_states: int, n_actions: int):
        self.n_states = n_states
        self.n_actions = n_actions
        # Count transitions
        self.counts = np.zeros((n_states, n_actions, n_states))
        self.rewards = np.zeros((n_states, n_actions))
        self.reward_counts = np.zeros((n_states, n_actions))

    def update(self, s, a, r, s2):
        self.counts[s, a, s2] += 1
        self.rewards[s, a] += r
        self.reward_counts[s, a] += 1

    def sample_transition(self, s: int, a: int) -> Tuple[int, float]:
        """Sample next state and reward from learned model."""
        cnt = self.counts[s, a]
        total = cnt.sum()
        if total == 0:
            # No data: return random
            return random.randint(0, self.n_states - 1), 0.0
        probs = cnt / total
        s2 = np.random.choice(self.n_states, p=probs)
        r = (self.rewards[s, a] / max(1, self.reward_counts[s, a]))
        return int(s2), float(r)

    def coverage(self) -> float:
        """Fraction of (s,a) pairs with at least one observation."""
        return float(np.mean(self.counts.sum(axis=2) > 0))


def run_mbrl(env: GridWorld1D, n_real_steps: int = 200,
              n_model_steps_per_real: int = 20,
              alpha: float = 0.1, gamma: float = 0.95,
              epsilon: float = 0.15) -> Tuple[List, List]:
    """Dyna-style MBRL: alternate real env steps and model-based planning."""
    model = DynamicsModel(env.n, 2)
    Q = np.zeros((env.n, 2))
    s = env.reset()
    real_returns = []
    model_coverage = []
    ep_ret = 0.0

    for step in range(n_real_steps):
        # Real step
        a = int(np.argmax(Q[s])) if random.random() > epsilon else random.randint(0, 1)
        s2, r, done = env.step(s, a)
        model.update(s, a, r, s2)
        Q[s, a] += alpha * (r + gamma * np.max(Q[s2]) * (1 - done) - Q[s, a])
        ep_ret += r
        if done:
            real_returns.append(ep_ret)
            ep_ret = 0.0
            s = env.reset()
        else:
            s = s2

        # Model-based planning (Dyna)
        for _ in range(n_model_steps_per_real):
            ms = random.randint(0, env.n - 1)
            ma = random.randint(0, 1)
            ms2, mr = model.sample_transition(ms, ma)
            mdone = (ms2 == env.goal)
            Q[ms, ma] += alpha * (mr + gamma * np.max(Q[ms2]) * (1 - mdone) - Q[ms, ma])

        model_coverage.append(model.coverage())

    return real_returns, model_coverage


def run_modelfree_qlearning(env: GridWorld1D, n_episodes: int = 200) -> List[float]:
    """Baseline: pure model-free Q-learning."""
    Q = np.zeros((env.n, 2))
    returns = []
    for _ in range(n_episodes):
        s = env.reset(); ep_ret = 0.0
        for _ in range(50):
            a = int(np.argmax(Q[s])) if random.random() > 0.15 else random.randint(0, 1)
            s2, r, done = env.step(s, a)
            Q[s, a] += 0.1 * (r + 0.95 * np.max(Q[s2]) * (1-done) - Q[s, a])
            s = s2; ep_ret += r
            if done: break
        returns.append(ep_ret)
    return returns


print("Running Dyna-style MBRL...")
mbrl_rets, coverage = run_mbrl(env, n_real_steps=300, n_model_steps_per_real=10)
mf_rets = run_modelfree_qlearning(env, n_episodes=300)

def smooth(x, w=10): return np.convolve(x, np.ones(w)/w, mode='valid')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('#0f1117')

ax = axes[0]
ax.set_facecolor('#0f1117')
x_mf = np.arange(len(smooth(mf_rets, 5)))
x_mb = np.arange(len(smooth(mbrl_rets, 5)))
ax.plot(x_mf, smooth(mf_rets, 5), color='#e94560', lw=2, label='Model-free Q-learning')
ax.plot(x_mb, smooth(mbrl_rets, 5), color='#ffcc02', lw=2, label='Dyna MBRL')
ax.set_xlabel('Episode'); ax.set_ylabel('Return')
ax.set_title('MBRL vs Model-Free: Sample Efficiency', color='white')
ax.legend(facecolor='#1a1a2e', labelcolor='white')
for item in [ax.xaxis.label, ax.yaxis.label] + ax.get_xticklabels() + ax.get_yticklabels():
    item.set_color('white')

ax = axes[1]
ax.set_facecolor('#0f1117')
ax.plot(smooth(coverage, 10), color='#80deea', lw=2)
ax.set_xlabel('Real Env Step'); ax.set_ylabel('Model Coverage (fraction of S×A)')
ax.set_title('Dynamics Model Coverage Growth', color='white')
for item in [ax.xaxis.label, ax.yaxis.label] + ax.get_xticklabels() + ax.get_yticklabels():
    item.set_color('white')

plt.suptitle('Model-Based RL (Dyna) vs Model-Free Q-Learning', color='white', fontsize=14)
plt.tight_layout(); plt.show()

<div style="background: #1b1b3a; padding: 16px; border-radius: 8px; border-left: 4px solid #ef9a9a;">
  <h2 style="color: #ef9a9a; margin: 0;">Часть 4 — Открытые направления исследований: обзор</h2>
</div>

Глава 17 обозревает ключевые открытые проблемы RL по состоянию на 2024–2025. Ниже визуализируем ландшафт.

In [ ]:
# Research landscape: effort vs maturity (qualitative)
topics = [
    ("Sample efficiency",        0.85, 0.60),
    ("Offline RL",               0.70, 0.55),
    ("Multi-agent RL",           0.65, 0.40),
    ("Reward specification",     0.55, 0.35),
    ("Exploration",              0.60, 0.50),
    ("Generalisation",           0.40, 0.30),
    ("Scalable RLHF",            0.90, 0.65),
    ("Model-based RL",           0.75, 0.55),
    ("Safe RL",                  0.50, 0.30),
    ("World models",             0.80, 0.50),
    ("Continual RL",             0.35, 0.20),
    ("Causal RL",                0.25, 0.15),
    ("Constitutional AI",        0.85, 0.60),
    ("Process reward models",    0.80, 0.55),
]

fig, ax = plt.subplots(figsize=(10, 7))
ax.set_facecolor('#0f1117'); fig.patch.set_facecolor('#0f1117')

colors_map = plt.cm.plasma
for name, activity, maturity in topics:
    color = colors_map(maturity)
    ax.scatter(activity, maturity, s=200, color=color, edgecolors='white', linewidths=0.5, zorder=5)
    ax.annotate(name, (activity, maturity), textcoords='offset points',
                xytext=(6, 2), fontsize=8.5, color='white')

ax.set_xlabel('Research Activity (relative)', fontsize=11)
ax.set_ylabel('Technical Maturity', fontsize=11)
ax.set_title('Open Problems in RL: Activity vs Maturity (2024)', color='white', fontsize=13)
ax.set_xlim(0.1, 1.0); ax.set_ylim(0.05, 0.80)

# Quadrant labels
ax.axvline(0.6, color='gray', linestyle=':', alpha=0.5)
ax.axhline(0.45, color='gray', linestyle=':', alpha=0.5)
ax.text(0.12, 0.70, 'High maturity\nlow activity', color='gray', fontsize=8)
ax.text(0.62, 0.70, 'High maturity\nhigh activity', color='#69f0ae', fontsize=8)
ax.text(0.12, 0.08, 'Low maturity\nlow activity', color='gray', fontsize=8)
ax.text(0.62, 0.08, 'Emerging frontier', color='#ffcc02', fontsize=8)

for item in [ax.xaxis.label, ax.yaxis.label] + ax.get_xticklabels() + ax.get_yticklabels():
    item.set_color('white')
plt.tight_layout(); plt.show()

## Открытые направления исследований — ключевые темы

### 1. Offline RL и Data-Driven RL
**Сложность**: ошибка экстраполяции — Q-значения для OOD-действий могут быть ошибочно высокими.
**Подходы**: Conservative Q-Learning (CQL), IQL, TD3+BC.
**Открытый вопрос**: как надёжно оценивать неопределённость без явных моделей плотности?

### 2. Спецификация вознаграждения и выравнивание
**Сложность**: предпочтения людей непоследовательны, контекстно-зависимы и трудно формализуемы.
**Подходы**: Constitutional AI, process reward models, debate.
**Открытый вопрос**: можно ли строить модели вознаграждения, не переобучающиеся и не позволяющие взлом вознаграждения в масштабе?

### 3. Model-Based RL и World Models
**Сложность**: накопление ошибок в многошаговых роллаутах ухудшает качество стратегии.
**Подходы**: Dreamer, MuZero, MBPO (короткие горизонты роллаутов).
**Открытый вопрос**: когда world-модель даёт чистый выигрыш над безмодельными методами?

### 4. Многоагентное и эмерджентное поведение
**Сложность**: нестационарность, присвоение заслуг, масштабируемость.
**Подходы**: QMIX, MADDPG, обучение на популяциях.
**Открытый вопрос**: можно ли надёжно направлять эмерджентное поведение к желаемым свойствам?

### 5. RL для рассуждения и инструментов
**Сложность**: разреженные вознаграждения, длинные горизонты, композиционные задачи.
**Подходы**: RLVR, process rewards, GRPO, LLM с инструментами.
**Открытый вопрос**: может ли RL-обученное рассуждение обобщаться за пределы обучающего распределения?

In [ ]:
# Summary comparison: key algorithms mentioned in Ch17
methods = ['Q-Learning\n(online)', 'Offline Q\n(random data)', 'Offline Q\n(expert data)',
            'Dyna MBRL', 'Decision\nTransformer']

# Evaluate each approach
def eval_approach(env, approach, n_runs=30):
    scores = []
    for _ in range(n_runs):
        if approach == 'online':
            Q = np.zeros((env.n, 2))
            for _ in range(150):
                s = env.reset()
                for _ in range(50):
                    a = int(np.argmax(Q[s])) if random.random() > 0.2 else random.randint(0,1)
                    s2, r, d = env.step(s, a)
                    Q[s,a] += 0.1*(r + 0.95*np.max(Q[s2])*(1-d) - Q[s,a])
                    s = s2
                    if d: break
            scores.append(evaluate_policy(env, Q, 30))
        elif approach in ('offline_random', 'offline_expert'):
            pt = 'random' if 'random' in approach else 'expert'
            ds = collect_dataset(env, pt, 2000)
            Q = offline_q_learning(env, ds, n_epochs=15)
            scores.append(evaluate_policy(env, Q, 30))
        elif approach == 'mbrl':
            rets, _ = run_mbrl(env, n_real_steps=150, n_model_steps_per_real=5)
            scores.append(np.mean(rets[-10:]) if rets else 0.0)
        elif approach == 'dt':
            scores.append(np.mean([dt_rollout(env, dt_fn, 9.0) for _ in range(10)]))
    return np.mean(scores), np.std(scores)

approaches = ['online', 'offline_random', 'offline_expert', 'mbrl', 'dt']
means, stds = [], []
for ap in approaches:
    m, s = eval_approach(env, ap, n_runs=15)
    means.append(m); stds.append(s)
    print(f"{ap:20s}: {m:.2f} ± {s:.2f}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.set_facecolor('#0f1117'); fig.patch.set_facecolor('#0f1117')
colors_ = ['#69f0ae', '#e94560', '#66bb6a', '#ffcc02', '#80deea']
bars = ax.bar(methods, means, yerr=stds, capsize=5,
              color=colors_, edgecolor='white', linewidth=0.8)
ax.set_ylabel('Mean Return')
ax.set_title('Algorithm Comparison: 1D GridWorld', color='white', fontsize=13)
for item in [ax.xaxis.label, ax.yaxis.label] + ax.get_xticklabels() + ax.get_yticklabels():
    item.set_color('white')
plt.tight_layout(); plt.show()

## Сводка и ключевые выводы

| Направление | Главная сложность | Перспективный подход |
|------|---------------|--------------------|
| Offline RL | Ошибка экстраполяции на OOD | CQL, IQL, консервативный пессимизм |
| Decision Transformer | Требует обусловливания достижимыми отдачами | Hindsight relabelling |
| Model-Based RL | Накопление ошибок модели | Dyna короткого горизонта, world models |
| Спецификация вознаграждения | Закон Гудхарта, взлом вознаграждения | Process rewards, constitutional AI |
| Multi-agent RL | Нестационарность, присвоение заслуг | Централизованное обучение, popular-методы |

Поле развивается быстро: граница между RL и крупномасштабным supervised-обучением размывается, поскольку RLVR и GRPO демонстрируют, что простые, масштабируемые проверяемые сигналы вознаграждения могут давать SOTA-способности к рассуждению.